# Entitiy Resolution PoC

Dieser PoC beschäftigt sich mit Entity Resolution (ER) mit Matryoshka Embeddings von `jina-embeddings-v4` und Hierarchical Navigable Small World (HNSW) für die ANN Suche als zu grunde liegende Technologien. Ziel ist die Maximierung der Konsistenz von oft unsauberen Daten.

## Setup

In [ ]:
import sys
!{sys.executable} -m pip install "transformers>=4.43,<5.0"
import transformers
print("Python:      ", sys.executable)
print("transformers:", transformers.__version__)
# ── 1. Environment Detection ──────────────────────────────────────────────────
import subprocess
import os


def in_kaggle():
    return "KAGGLE_KERNEL_RUN_TYPE" in os.environ

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["transformers", "faker", "matplotlib", "seaborn", "scikit-learn", "pandas", "numpy"]:
    install(pkg)

try:
    import torch
except ImportError:
    install("torch")

try:
    import faiss
except ImportError:
    if in_kaggle():
        subprocess.check_call(["conda", "install", "-y", "-c", "pytorch", "-c", "nvidia", "faiss-gpu"])
    else:
        install("faiss-cpu")  # Mac M4 → faiss-cpu

# ── 2. Device Setup ───────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")       # Kaggle T4
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")        # Mac M4
else:
    DEVICE = torch.device("cpu")

print(f"Environment : {'Kaggle' if in_kaggle() else 'Local'}")
print(f"Device      : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU         : {torch.cuda.get_device_name(0)}")

# ── 3. FAISS Index Helper (GPU-aware) ─────────────────────────────────────────
import faiss

def make_faiss_index(dim: int) -> faiss.Index:
    """Gibt GPU-Index auf Kaggle zurück, CPU-Index sonst."""
    index = faiss.IndexFlatL2(dim)
    if DEVICE.type == "cuda":
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
    return index

import time
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

## Synthetische Datenerstellung

In [ ]:
from DataProvider import data_generator

df_base = data_generator.generate_base_data(n=10)
df_noisy = data_generator.generate_noisy_duplicates(df_base)
print(f"Generiert: {len(df_base)} Originale, {len(df_noisy)} verrauschte Duplikate.")

## Erstellung der Embeddings

In [ ]:
from EmbeddingPipe import pipeline

print("Erstelle Embeddings für Originale...")
base_embeddings_2048 = pipeline.encode(df_base["text"].tolist())
print("Erstelle Embeddings für Verrauschte...")
noisy_embeddings_2048 = pipeline.encode(df_noisy["text"].tolist())
print(f"Storage Original Embeddings: {int(base_embeddings_2048.nbytes  / 1000)}KB\nStorage Noisy Embeddings: {int(noisy_embeddings_2048.nbytes / 1000)}KB")

## Indizierung & Evaluation

In [ ]:
from EmbeddingPipe import truncate_and_normalize

k = 5

def evaluate_dimension(dim, base_emb_full, noisy_emb_full, ground_truth_ids) -> dict:
    print(f"Evaluiere Dimension: {dim}")

    # Truncation
    base_emb = truncate_and_normalize(base_emb_full, dim)
    noisy_emb = truncate_and_normalize(noisy_emb_full, dim)

    # HNSW
    index = faiss.IndexHNSWFlat(dim, 32)
    index.add(base_emb)

    start_time = time.perf_counter()
    distances, indices = index.search(noisy_emb, k)
    latency_ms = ((time.perf_counter() - start_time) * 1000) / len(noisy_emb)

    # Metrics
    recall_at_1 = np.mean([1 if gt == idx[0] else 0 for gt, idx in zip(ground_truth_ids, indices)])
    recall_at_5 = np.mean([1 if gt in idx else 0 for gt, idx in zip(ground_truth_ids, indices)])

    # Information
    base_full_norm = truncate_and_normalize(base_emb_full, 2048)
    noisy_full_norm = truncate_and_normalize(noisy_emb_full, 2048)

    sims_full = np.sum(base_full_norm * noisy_full_norm, axis=1)
    sims_trunc = np.sum(base_emb * noisy_emb, axis=1)
    info_retention, _ = pearsonr(sims_full, sims_trunc)
    
    return {
        "Dimension": dim,
        "Recall@1": recall_at_1,
        "Recall@5": recall_at_5,
        "Latency (ms)": latency_ms,
        "Info Retention (Corr)": info_retention
    }

## Visualisierung & Reporting

In [ ]:
def plot_results(results_df):
    sns.set_theme(style="whitegrid")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Scree-Plot (Informationserhalt vs. Dimension)
    ax1.plot(results_df["Dimension"], results_df["Info Retention (Corr)"], marker='o', color='b', linewidth=2)
    ax1.set_xscale('log')
    ax1.set_xticks([128, 512, 2048])
    ax1.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax1.set_title("Scree-Plot: Informationserhalt bei Dimensionsreduktion")
    ax1.set_xlabel("Vektor-Dimension (Matryoshka)")
    ax1.set_ylabel("Korrelation mit 2048d-Repräsentation")
    ax1.set_ylim(0.8, 1.05)
    
    # Plot 2: Trade-off Bar Chart (Latenz vs. Recall@1)
    ax2_twin = ax2.twinx()
    width = 0.3
    x = np.arange(len(results_df["Dimension"]))
    
    bars1 = ax2.bar(x - width/2, results_df["Recall@1"], width, label='Recall@1 (Präzision)', color='#2ca02c')
    bars2 = ax2_twin.bar(x + width/2, results_df["Latency (ms)"], width, label='Latenz (ms)', color='#d62728')
    
    ax2.set_xticks(x)
    ax2.set_xticklabels(results_df["Dimension"])
    ax2.set_xlabel("Dimension")
    ax2.set_ylabel("Recall@1 Score", color='#2ca02c')
    ax2_twin.set_ylabel("Such-Latenz (ms / Query)", color='#d62728')
    ax2.set_title("Trade-off: Suchpräzision vs. Latenz (FAISS HNSW)")
    
    # Legenden zusammenführen
    lines, labels = ax2.get_legend_handles_labels()
    lines2, labels2 = ax2_twin.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='lower right')
    
    plt.tight_layout()
    plt.show()

## Ausführung

In [ ]:
dimensions = [128, 512, 2048]
results = []

for dim in dimensions:
    res = evaluate_dimension(
        dim=dim,
        base_emb_full=base_embeddings_2048,
        noisy_emb_full=noisy_embeddings_2048,
        ground_truth_ids=df_noisy['ground_truth_id'].tolist()
    )
    results.append(res)
        
results_df = pd.DataFrame(results)
print("\n--- Evaluationsergebnisse ---")
print(results_df.to_string(index=False))

plot_results(results_df)